# F1 Two-Axis Ablation Report

This notebook is the follow-up report template for comparing two harness axes:

- Action Gating: baseline tool exposure vs terminal-only/file-only variants
- Transition Control: terminal timeout 60s vs 20s

It reads `benchmark/results/f1_*.jsonl` and uses only the Python standard library. It is intentionally runnable before all future result files exist; missing cells in the 2x2 grid are reported as pending.

## Planned Result Files

Current Action Gating files:

- `benchmark/results/f1_baseline.jsonl`
- `benchmark/results/f1_file_only.jsonl`
- `benchmark/results/f1_terminal_only.jsonl`

New Transition Control files to run next:

- `benchmark/results/f1_transition_timeout_20.jsonl`
- `benchmark/results/f1_terminal_timeout_20.jsonl`

The 2x2 grid is: action space (`baseline`, `terminal_only`) x terminal timeout (`60`, `20`).

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent

RESULTS_DIR = ROOT / "benchmark" / "results"
RESULT_FILES = sorted(RESULTS_DIR.glob("f1_*.jsonl"))

def percentile(values: list[float], pct: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return ordered[0]
    rank = (len(ordered) - 1) * pct
    lower = int(rank)
    upper = min(lower + 1, len(ordered) - 1)
    weight = rank - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

def mean(values: list[float]) -> float | None:
    return sum(values) / len(values) if values else None

def config_name_from_record(rec: dict, path: Path) -> str:
    config_path = rec.get("config_path")
    if config_path:
        return Path(config_path).stem
    return path.stem.removeprefix("f1_")

def classify_config(config: str) -> dict[str, object]:
    if config == "baseline":
        return {"action_axis": "baseline", "transition_timeout": 60, "grid_key": "baseline@60"}
    if config == "action_terminal_only":
        return {"action_axis": "terminal_only", "transition_timeout": 60, "grid_key": "terminal_only@60"}
    if config == "action_file_only":
        return {"action_axis": "file_only", "transition_timeout": 60, "grid_key": "file_only@60"}
    if config == "transition_timeout_20":
        return {"action_axis": "baseline", "transition_timeout": 20, "grid_key": "baseline@20"}
    if config == "action_terminal_timeout_20":
        return {"action_axis": "terminal_only", "transition_timeout": 20, "grid_key": "terminal_only@20"}
    return {"action_axis": "unknown", "transition_timeout": None, "grid_key": config}

rows = []
for path in RESULT_FILES:
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        metrics = rec.get("metrics") or {}
        config = config_name_from_record(rec, path)
        axes = classify_config(config)
        rows.append({
            "config": config,
            "file": path.name,
            "task_id": rec.get("task_id"),
            "seed": rec.get("seed"),
            "q_pass": metrics.get("q_oracle_pass") is True,
            "latency_s": float(metrics.get("l_wall_clock_sec") or 0),
            "input_tokens": int(metrics.get("c_input_tokens") or 0),
            "output_tokens": int(metrics.get("c_output_tokens") or 0),
            "api_calls": int(rec.get("api_calls") or 0),
            "stop_reason": rec.get("stop_reason"),
            **axes,
        })

print("loaded records:", len(rows))
print("files:", ", ".join(path.name for path in RESULT_FILES))


In [ ]:
summary = []
for config in sorted({row["config"] for row in rows}):
    subset = [row for row in rows if row["config"] == config]
    latencies = [row["latency_s"] for row in subset]
    summary.append({
        "config": config,
        "action_axis": subset[0]["action_axis"],
        "transition_timeout": subset[0]["transition_timeout"],
        "runs": len(subset),
        "passes": sum(row["q_pass"] for row in subset),
        "pass_rate": mean([1.0 if row["q_pass"] else 0.0 for row in subset]),
        "latency_p50": percentile(latencies, 0.50),
        "latency_p95": percentile(latencies, 0.95),
        "latency_max": max(latencies),
        "input_tokens_mean": mean([row["input_tokens"] for row in subset]),
        "output_tokens_mean": mean([row["output_tokens"] for row in subset]),
        "api_calls_mean": mean([row["api_calls"] for row in subset]),
        "stop_reasons": dict(Counter(row["stop_reason"] for row in subset)),
    })

print("config                 action          timeout  runs  pass  p50_s   p95_s   max_s   in_tok  out_tok  api")
print("---------------------  --------------  -------  ----  ----  ------  ------  ------  ------  -------  ----")
for row in summary:
    print(
        f"{row['config']:<21}  {row['action_axis']:<14}  {str(row['transition_timeout']):>7}  "
        f"{row['runs']:>4}  {row['passes']:>4}  {row['latency_p50']:>6.2f}  {row['latency_p95']:>6.2f}  "
        f"{row['latency_max']:>6.2f}  {row['input_tokens_mean']:>6.1f}  {row['output_tokens_mean']:>7.1f}  "
        f"{row['api_calls_mean']:>4.2f}"
    )


In [ ]:
grid = {(row["action_axis"], row["transition_timeout"]): row for row in summary}
planned = [("baseline", 60), ("baseline", 20), ("terminal_only", 60), ("terminal_only", 20)]

print("2x2 grid status")
for key in planned:
    row = grid.get(key)
    if row is None:
        print(f"{key[0]:<13} timeout={key[1]:>2}: PENDING")
    else:
        print(
            f"{key[0]:<13} timeout={key[1]:>2}: pass={row['passes']}/{row['runs']} "
            f"p50={row['latency_p50']:.2f}s p95={row['latency_p95']:.2f}s "
            f"input_mean={row['input_tokens_mean']:.1f}"
        )

print("\nDeltas once timeout=20 cells are present:")
for action in ("baseline", "terminal_only"):
    high = grid.get((action, 60))
    low = grid.get((action, 20))
    if not high or not low:
        print(f"{action}: pending")
        continue
    print(
        f"{action}: p95_delta={low['latency_p95'] - high['latency_p95']:.2f}s, "
        f"pass_delta={low['pass_rate'] - high['pass_rate']:.3f}, "
        f"input_delta={(low['input_tokens_mean'] / high['input_tokens_mean'] - 1) * 100:.2f}%"
    )


## Run Commands

The configs now default to `seeds_per_task: 5`. Run these from repo root after setting `HERMES_BENCH_API_KEY`, `HERMES_HOME`, `HOME`, `UV_CACHE_DIR`, and `PYTHONUNBUFFERED` as in the previous report.

```bash
uv run --project hermes_v-0-10-0 python -u benchmark/run_benchmark.py \
  --family F1_code_qa \
  --config /home/nan/c_project/hfa/benchmark/configs/baseline.yaml \
  --runner hermes_direct \
  --out /home/nan/c_project/hfa/benchmark/results/f1_baseline.jsonl

uv run --project hermes_v-0-10-0 python -u benchmark/run_benchmark.py \
  --family F1_code_qa \
  --config /home/nan/c_project/hfa/benchmark/configs/action_file_only.yaml \
  --runner hermes_direct \
  --out /home/nan/c_project/hfa/benchmark/results/f1_file_only.jsonl

uv run --project hermes_v-0-10-0 python -u benchmark/run_benchmark.py \
  --family F1_code_qa \
  --config /home/nan/c_project/hfa/benchmark/configs/action_terminal_only.yaml \
  --runner hermes_direct \
  --out /home/nan/c_project/hfa/benchmark/results/f1_terminal_only.jsonl

uv run --project hermes_v-0-10-0 python -u benchmark/run_benchmark.py \
  --family F1_code_qa \
  --config /home/nan/c_project/hfa/benchmark/configs/transition_timeout_20.yaml \
  --runner hermes_direct \
  --out /home/nan/c_project/hfa/benchmark/results/f1_transition_timeout_20.jsonl

uv run --project hermes_v-0-10-0 python -u benchmark/run_benchmark.py \
  --family F1_code_qa \
  --config /home/nan/c_project/hfa/benchmark/configs/action_terminal_timeout_20.yaml \
  --runner hermes_direct \
  --out /home/nan/c_project/hfa/benchmark/results/f1_terminal_timeout_20.jsonl
```
